In [40]:
# ============================================================
# Arriba parsing (simple + support)
# Output: sample, caller, fusion, support
# Keep: high + medium confidence (drop low)
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

ARRIBA_ROOT = Path("/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/ARRIBA")
REPORT_ROOT = Path("/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports")
REPORT_ROOT.mkdir(parents=True, exist_ok=True)


def list_sample_dirs(root: Path) -> Dict[str, Path]:
    """Immediate subfolders as samples: {sample: path}."""
    if not root.exists():
        return {}
    return {p.name: p for p in sorted(root.iterdir()) if p.is_dir()}


def read_arriba_tsv_keep_header(path: Path) -> pd.DataFrame:
    """
    Arriba header begins with '#gene1'. Do NOT use comment='#' because that drops the header.
    We locate the header line, then read from there.
    """
    header_idx = None
    with path.open("r", encoding="utf-8", errors="replace") as fh:
        for i, line in enumerate(fh):
            if line.startswith("#gene1\t") or line.startswith("#gene1 "):
                header_idx = i
                break
    if header_idx is None:
        raise ValueError(f"Could not find Arriba header '#gene1' in: {path}")

    df = pd.read_csv(path, sep="\t", header=0, skiprows=header_idx, low_memory=False)
    df.columns = [c.lstrip("#").strip() for c in df.columns]
    return df


def parse_arriba_one_sample(sample: str, sample_dir: Path) -> Tuple[pd.DataFrame, dict]:
    """
    Returns:
      - calls_df with columns: sample, caller, fusion, support (ONLY high+medium)
      - stats dict for sanity table (ALL rows in the file)
    """
    files = sorted(sample_dir.glob("*.arriba.fusions.tsv"))
    if not files:
        return (
            pd.DataFrame(columns=["sample", "caller", "fusion", "support"]),
            {"Sample": sample, "Total_Number_Of_Fusion": 0, "High": 0, "Medium": 0, "Low": 0},
        )

    # pick newest if multiple
    f = max(files, key=lambda p: p.stat().st_mtime)

    df_all = read_arriba_tsv_keep_header(f)

    if "confidence" not in df_all.columns:
        raise ValueError(f"Arriba file missing 'confidence' column: {f}")

    conf = df_all["confidence"].astype(str).str.lower()

    # counts for reporting (ALL rows in file)
    n_total = int(df_all.shape[0])
    n_high = int((conf == "high").sum())
    n_med  = int((conf == "medium").sum())
    n_low  = int((conf == "low").sum())

    # keep only high + medium
    df = df_all[conf.isin({"high", "medium"})].copy()

    rows: List[dict] = []
    for _, r in df.iterrows():
        g1 = str(r.get("gene1", "")).strip()
        g2 = str(r.get("gene2", "")).strip()
        fusion = f"{g1}--{g2}"

        sr1 = pd.to_numeric(r.get("split_reads1", pd.NA), errors="coerce")
        sr2 = pd.to_numeric(r.get("split_reads2", pd.NA), errors="coerce")
        dm  = pd.to_numeric(r.get("discordant_mates", pd.NA), errors="coerce")

        support = None
        if pd.notna(sr1) or pd.notna(sr2) or pd.notna(dm):
            support = float(
                (sr1 if pd.notna(sr1) else 0)
                + (sr2 if pd.notna(sr2) else 0)
                + (dm  if pd.notna(dm)  else 0)
            )

        # skip totally empty fusion like "--"
        if fusion == "--":
            continue

        rows.append({
            "sample": sample,
            "caller": "arriba",
            "fusion": fusion,
            "support": support
        })

    calls_df = pd.DataFrame(rows, columns=["sample", "caller", "fusion", "support"])

    stats = {
        "Sample": sample,
        "Total_Number_Of_Fusion": n_total,
        "High": n_high,
        "Medium": n_med,
        "Low": n_low,
    }
    return calls_df, stats


def main():
    arriba_samples = list_sample_dirs(ARRIBA_ROOT)

    all_calls = []
    sanity_rows = []

    for s, sdir in arriba_samples.items():
        calls_df, stats = parse_arriba_one_sample(s, sdir)
        all_calls.append(calls_df)
        sanity_rows.append(stats)

    calls_all_df = pd.concat(all_calls, ignore_index=True) if all_calls else pd.DataFrame(
        columns=["sample", "caller", "fusion", "support"]
    )
    sanity_df = pd.DataFrame(sanity_rows).sort_values("Sample") if sanity_rows else pd.DataFrame()

    print("Arriba samples:", len(arriba_samples))
    print("Arriba calls rows (high+medium only):", int(calls_all_df.shape[0]))

    if not sanity_df.empty:
        print(sanity_df[["Sample", "Total_Number_Of_Fusion", "High", "Medium", "Low"]].to_string(index=False))
    else:
        print("(no samples found)")

    # Save ONLY the 4 columns you want
    calls_all_df.to_csv(REPORT_ROOT / "arriba.calls.high_medium.tsv", sep="\t", index=False)
    sanity_df.to_csv(REPORT_ROOT / "arriba.sanity.tsv", sep="\t", index=False)


if __name__ == "__main__":
    main()


Arriba samples: 31
Arriba calls rows (high+medium only): 725
   Sample  Total_Number_Of_Fusion  High  Medium  Low
 TP03-117                      38     9       8   21
 TP04-104                      98    34      18   46
TP05-1063                      50    10      16   24
 TP05-232                      73    18      18   37
 TP05-668                      77    23      14   40
 TP05-726                      31     3       7   21
 TP05-791                      55    19      10   26
 TP07-197                      36     2       3   31
 TP07-626                      17     2       2   13
TP08-1319                      53    11      12   30
TP08-2202                      50    21       6   23
 TP08-356                     119    20      31   68
 TP08-370                      30     2       2   26
 TP08-517                      32    11       5   16
 TP09-746                      28    13       1   14
   TP09-9                      41    11       8   22
 TP10-705                      32     

In [41]:
# ============================================================
# FusionCatcher parsing
# - Skips entries annotated as 'false positive'
# ============================================================

from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

FUSIONCATCHER_ROOT = Path("/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionCatcher")
REPORT_ROOT = Path("/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports")
REPORT_ROOT.mkdir(parents=True, exist_ok=True)


def list_sample_dirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {p.name: p for p in sorted(root.iterdir()) if p.is_dir()}


def parse_fusioncatcher_one_sample(sample: str, sample_dir: Path) -> pd.DataFrame:
    """
    Parse FusionCatcher summary_candidate_fusions.txt

    Returns DataFrame with columns:
      sample, caller, fusion, support
    support is NA because this file format doesn't provide a consistent numeric support.
    """
    f = sample_dir / "summary_candidate_fusions.txt"
    if not f.exists():
        return pd.DataFrame(columns=["sample", "caller", "fusion", "support"])

    rows: List[dict] = []
    with f.open("r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.rstrip("\n")

            # Lines look like: "* GENE1--GENE2 (something...)"
            m = re.match(r"^\s*\*\s+(\S+)", line)
            if not m:
                continue

            fusion_token = m.group(1).strip()

            # Extract annotation in parentheses
            annot = ""
            if "(" in line and ")" in line:
                annot = line[line.find("(") + 1 : line.rfind(")")].strip().lower()

            # Skip false positives
            if "false positive" in annot:
                continue

            fusion = fusion_token.strip()

            # Support is not available in this summary list -> keep as NA
            rows.append({
                "sample": sample,
                "caller": "fusioncatcher",
                "fusion": fusion,
                "support": pd.NA,
            })

    return pd.DataFrame(rows, columns=["sample", "caller", "fusion", "support"])


def fusioncatcher_count_total_and_falsepos(sample_dir: Path) -> Tuple[int, int, bool]:
    f = sample_dir / "summary_candidate_fusions.txt"
    if not f.exists():
        return 0, 0, False

    total = 0
    falsepos = 0
    with f.open("r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            m = re.match(r"^\s*\*\s+(\S+)", line)
            if not m:
                continue
            total += 1

            annot = ""
            if "(" in line and ")" in line:
                annot = line[line.find("(") + 1 : line.rfind(")")].strip().lower()
            if "false positive" in annot:
                falsepos += 1

    return total, falsepos, True


def main():
    fc_samples = list_sample_dirs(FUSIONCATCHER_ROOT)

    all_calls = []
    sanity_rows = []

    for s, sdir in fc_samples.items():
        calls_df = parse_fusioncatcher_one_sample(s, sdir)
        all_calls.append(calls_df)

        total, falsepos, present = fusioncatcher_count_total_and_falsepos(sdir)
        sanity_rows.append({
            "Sample": s,
            "FusionCatcher_total_listed": total,
            "FusionCatcher_false_positive_listed": falsepos,
            "FusionCatcher_fusions_kept": int(calls_df.shape[0]),
        })

    calls_all_df = pd.concat(all_calls, ignore_index=True) if all_calls else pd.DataFrame(
        columns=["sample", "caller", "fusion", "support"]
    )
    sanity_df = pd.DataFrame(sanity_rows).sort_values("Sample") if sanity_rows else pd.DataFrame()

    print("FusionCatcher samples:", len(fc_samples))
    print("FusionCatcher calls rows (kept):", int(calls_all_df.shape[0]))

    if not sanity_df.empty:
        print(sanity_df[[
            "Sample",
            "FusionCatcher_total_listed",
            "FusionCatcher_false_positive_listed",
            "FusionCatcher_fusions_kept"
        ]].to_string(index=False))
    else:
        print("(no samples found)")

    # Save
    calls_all_df.to_csv(REPORT_ROOT / "fusioncatcher.calls.kept.tsv", sep="\t", index=False)
    sanity_df.to_csv(REPORT_ROOT / "fusioncatcher.sanity.tsv", sep="\t", index=False)


if __name__ == "__main__":
    main()


FusionCatcher samples: 31
FusionCatcher calls rows (kept): 721
   Sample  FusionCatcher_total_listed  FusionCatcher_false_positive_listed  FusionCatcher_fusions_kept
 TP03-117                          34                                   20                          14
 TP04-104                          60                                   32                          28
TP05-1063                          47                                   28                          19
 TP05-232                          52                                   19                          33
 TP05-668                          60                                   31                          29
 TP05-726                          37                                   33                           4
 TP05-791                          50                                   28                          22
 TP07-197                          53                                   36                          17
 TP07-626 

In [42]:
# ============================================================
# STAR-Fusion parsing (simple + support)
# Output: sample, caller, fusion, support
# Filter: FFPM >= 0.1
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

STARFUSION_ROOT = Path("/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/StarFusion")
REPORT_ROOT = Path("/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports")
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

STARFUSION_MIN_FFPM: Optional[float] = 0.1


def list_sample_dirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {p.name: p for p in sorted(root.iterdir()) if p.is_dir()}


def clean_gene_from_starfusion(g: str) -> str:
    """STAR-Fusion uses 'SYMBOL^ENSG...' in LeftGene/RightGene."""
    if pd.isna(g):
        return ""
    s = str(g).strip()
    if not s or s.lower() == "nan":
        return ""
    return s.split("^", 1)[0].strip()


def read_starfusion_tsv_keep_header(path: Path) -> pd.DataFrame:
    """
    STAR-Fusion header begins with '#FusionName' so we must NOT drop '#' lines.
    """
    df = pd.read_csv(path, sep="\t", header=0, low_memory=False)
    df.columns = [c.lstrip("#").strip() for c in df.columns]
    return df


def parse_starfusion_one_sample(sample: str, sample_dir: Path) -> Tuple[pd.DataFrame, int, int]:
    f = sample_dir / "star-fusion.fusion_predictions.tsv"
    if not f.exists():
        return pd.DataFrame(columns=["sample", "caller", "fusion", "support"]), 0, 0

    # Count fusion rows in file (exclude header/comment/blank)
    file_rows = 0
    with f.open("r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if not line.strip():
                continue
            if line.startswith("#"):
                continue
            file_rows += 1

    df = read_starfusion_tsv_keep_header(f)

    # Apply FFPM filter
    if STARFUSION_MIN_FFPM is not None and "FFPM" in df.columns:
        ffpm = pd.to_numeric(df["FFPM"], errors="coerce")
        df = df[ffpm >= STARFUSION_MIN_FFPM].copy()

    out_rows: List[dict] = []
    skipped_empty = 0

    for _, r in df.iterrows():
        fusion = ""

        # Prefer FusionName if present
        fusionname = str(r.get("FusionName", "")).strip()
        if fusionname and fusionname.lower() != "nan" and "--" in fusionname:
            g1, g2 = fusionname.split("--", 1)
            g1 = clean_gene_from_starfusion(g1.strip())
            g2 = clean_gene_from_starfusion(g2.strip())
            if g1 and g2:
                fusion = f"{g1}--{g2}"

        # Fallback: LeftGene/RightGene
        if not fusion:
            lg = clean_gene_from_starfusion(r.get("LeftGene", ""))
            rg = clean_gene_from_starfusion(r.get("RightGene", ""))
            if lg and rg:
                fusion = f"{lg}--{rg}"

        if not fusion or fusion == "--":
            skipped_empty += 1
            continue

        # support = JunctionReadCount + SpanningFragCount
        jr = pd.to_numeric(r.get("JunctionReadCount", pd.NA), errors="coerce")
        sf = pd.to_numeric(r.get("SpanningFragCount", pd.NA), errors="coerce")
        support = None
        if pd.notna(jr) or pd.notna(sf):
            support = float((jr if pd.notna(jr) else 0) + (sf if pd.notna(sf) else 0))

        out_rows.append({
            "sample": sample,
            "caller": "starfusion",
            "fusion": fusion,
            "support": support
        })

    out_df = pd.DataFrame(out_rows, columns=["sample", "caller", "fusion", "support"])
    return out_df, file_rows, skipped_empty


def main():
    sf_dirs_all = list_sample_dirs(STARFUSION_ROOT)
    sf_samples = {s: d for s, d in sf_dirs_all.items()
                  if (d / "star-fusion.fusion_predictions.tsv").exists()}

    all_calls = []
    sanity_rows = []
    total_skipped = 0

    for s, sdir in sf_samples.items():
        calls_df, file_rows, skipped = parse_starfusion_one_sample(s, sdir)
        all_calls.append(calls_df)
        total_skipped += skipped

        sanity_rows.append({
            "Sample": s,
            "StarFusion_fusions_in_file": file_rows,
            "StarFusion_fusions_kept": int(calls_df.shape[0]),
            "StarFusion_skipped_empty": skipped,
        })

    calls_all_df = pd.concat(all_calls, ignore_index=True) if all_calls else pd.DataFrame(
        columns=["sample", "caller", "fusion", "support"]
    )
    sanity_df = pd.DataFrame(sanity_rows).sort_values("Sample") if sanity_rows else pd.DataFrame()

    print("STAR-Fusion samples:", len(sf_samples))
    print("STAR-Fusion calls rows (after FFPM >= 0.1):", int(calls_all_df.shape[0]))
    print("Total skipped (empty/unparseable):", total_skipped)

    if not sanity_df.empty:
        print(sanity_df.to_string(index=False))
    else:
        print("(no samples found)")

    calls_all_df.to_csv(REPORT_ROOT / "starfusion.calls.filtered.tsv", sep="\t", index=False)
    sanity_df.to_csv(REPORT_ROOT / "starfusion.sanity.tsv", sep="\t", index=False)


if __name__ == "__main__":
    main()


STAR-Fusion samples: 31
STAR-Fusion calls rows (after FFPM >= 0.1): 235
Total skipped (empty/unparseable): 0
   Sample  StarFusion_fusions_in_file  StarFusion_fusions_kept  StarFusion_skipped_empty
 TP03-117                           4                        4                         0
 TP04-104                          17                       17                         0
TP05-1063                           6                        6                         0
 TP05-232                          17                       17                         0
 TP05-668                          17                       17                         0
 TP05-726                           3                        3                         0
 TP05-791                          16                       16                         0
 TP07-197                           3                        3                         0
 TP07-626                           3                        3                         0
T

In [4]:
# RNA fusion caller integration (Arriba, FusionCatcher, STAR-Fusion)
#!/usr/bin/env python3
# ============================================================
# RNA fusion caller integration (Arriba, FusionCatcher, STAR-Fusion)
# Inputs: per-caller TSVs with columns: sample, caller, fusion, support
# Outputs:
#   - cohort.fusionreport.tsv
#   - cohort.filtered.fusionreport.tsv   (>=2 callers)
#   - caller_sample_sanity.tsv
#   - fusion_recurrence_across_samples.tsv
# ============================================================

from __future__ import annotations

from pathlib import Path
import pandas as pd

REPORT_ROOT = Path("/ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports")

ARRIBA_CALLS = REPORT_ROOT / "arriba.calls.high_medium.tsv"
FC_CALLS     = REPORT_ROOT / "fusioncatcher.calls.kept.tsv"
SF_CALLS     = REPORT_ROOT / "starfusion.calls.filtered.tsv"

OUT_ALL   = REPORT_ROOT / "cohort.fusionreport.tsv"
OUT_FILT  = REPORT_ROOT / "cohort.filtered.fusionreport.tsv"
OUT_SANITY_SAMPLE = REPORT_ROOT / "caller_sample_sanity.tsv"
OUT_RECURRENCE = REPORT_ROOT / "fusion_recurrence_across_samples.tsv"

REQUIRED_COLS = ["sample", "caller", "fusion", "support"]


def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip() for c in df.columns]
    # normalize required columns to lowercase
    colmap = {c: c.lower() for c in df.columns}
    df = df.rename(columns=colmap)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}. Found columns: {list(df.columns)}")

    # strip strings
    df["sample"] = df["sample"].astype(str).str.strip()
    df["caller"] = df["caller"].astype(str).str.strip().str.lower()
    df["fusion"] = df["fusion"].astype(str).str.strip()

    # coerce support (can be empty)
    df["support"] = pd.to_numeric(df["support"], errors="coerce")

    # drop truly empty sample/fusion rows
    df = df[(df["sample"] != "") & (df["fusion"] != "") & (df["fusion"] != "--")].copy()
    return df


def load_calls(path: Path, expected_caller: str) -> pd.DataFrame:
    if not path.exists():
        print(f"[WARN] Missing file: {path}")
        return pd.DataFrame(columns=REQUIRED_COLS)

    df = pd.read_csv(path, sep="\t", low_memory=False)
    df = normalize_df(df)

    # enforce caller label (in case file has mixed/odd)
    df["caller"] = expected_caller.lower()

    print(f"[LOAD] {expected_caller:13s} rows={len(df)}")
    return df


def build_caller_sample_sanity(arriba: pd.DataFrame, fc: pd.DataFrame, sf: pd.DataFrame) -> pd.DataFrame:
    def summarize(df: pd.DataFrame, colname: str) -> pd.DataFrame:
        if df.empty:
            return pd.DataFrame(columns=["Sample", colname])
        return (
            df.groupby("sample").size()
            .reset_index(name=colname)
            .rename(columns={"sample": "Sample"})
        )

    sa = summarize(arriba, "arriba_calls")
    sb = summarize(fc, "fusioncatcher_calls")
    sc = summarize(sf, "starfusion_calls")

    sanity = sa.merge(sb, on="Sample", how="outer").merge(sc, on="Sample", how="outer")
    for c in ["arriba_calls", "fusioncatcher_calls", "starfusion_calls"]:
        sanity[c] = sanity.get(c, 0)
        sanity[c] = sanity[c].fillna(0).astype(int)

    return sanity.sort_values("Sample").reset_index(drop=True)


def main():
    arriba = load_calls(ARRIBA_CALLS, "arriba")
    fc     = load_calls(FC_CALLS, "fusioncatcher")
    sf     = load_calls(SF_CALLS, "starfusion")

    # ----------------------------
    # caller_sample_sanity.tsv
    # ----------------------------
    sanity_sample = build_caller_sample_sanity(arriba, fc, sf)
    sanity_sample.to_csv(OUT_SANITY_SAMPLE, sep="\t", index=False)
    print(f"\nWrote: {OUT_SANITY_SAMPLE} | rows: {sanity_sample.shape[0]}")

    # ----------------------------
    # Combine calls
    # ----------------------------
    all_calls = pd.concat([arriba, fc, sf], ignore_index=True)
    if all_calls.empty:
        raise SystemExit("[ERROR] No calls found across callers.")

    # ----------------------------
    # Canonical fusion label for concordance
    # (simple, orientation-insensitive, no BAD_STRINGS logic)
    # ----------------------------
    def canon_fusion(f: str) -> str:
        f = (f or "").strip()
        if "--" not in f:
            return f
        g1, g2 = [x.strip() for x in f.split("--", 1)]
        if not g1 and not g2:
            return ""
        if not g1:
            return g2
        if not g2:
            return g1
        return "--".join(sorted([g1, g2]))

    all_calls["FusionCanon"] = all_calls["fusion"].map(canon_fusion)
    all_calls = all_calls[all_calls["FusionCanon"] != ""].copy()

    # ----------------------------
    # Merge to unique (Sample, FusionCanon)
    # Caller list + caller count
    # ----------------------------
    cohort = (
        all_calls.groupby(["sample", "FusionCanon"])
        .agg(
            Caller=("caller", lambda x: ",".join(sorted(set(x)))),
            N_callers=("caller", lambda x: len(set(x))),
        )
        .reset_index()
        .rename(columns={"sample": "Sample", "FusionCanon": "Fusion"})
    )

    # ----------------------------
    # Fusion_count = number of samples containing this fusion
    # ----------------------------
    fusion_to_n_samples = (
        cohort.groupby("Fusion")["Sample"].nunique()
        .rename("Fusion_count")
        .reset_index()
    )
    cohort = cohort.merge(fusion_to_n_samples, on="Fusion", how="left")
    cohort["Fusion_count"] = cohort["Fusion_count"].fillna(0).astype(int)

    # ----------------------------
    # cohort.fusionreport.tsv (collapse duplicates fusion within each sample)
    # ----------------------------
    cohort_all_out = cohort[["Fusion", "Sample", "Fusion_count", "Caller"]].sort_values(
        ["Sample", "Fusion"], ascending=[True, True]
    )
    cohort_all_out.to_csv(OUT_ALL, sep="\t", index=False)
    print(f"Wrote: {OUT_ALL} | rows: {cohort_all_out.shape[0]}")

    # ----------------------------
    # cohort.filtered.fusionreport.tsv (>=2 callers)
    # (fusion-level summary: one row per Fusion, samples collapsed)
    # ----------------------------
    cohort_filt = cohort[cohort["N_callers"] >= 2].copy()

    cohort_filt_fusion_level = (
        cohort_filt.groupby("Fusion")
        .agg(
            Sample=("Sample", lambda x: ",".join(sorted(set(x)))),
            Fusion_count=("Sample", "nunique"),  # number of samples with this concordant fusion
            Caller=("Caller", lambda x: ",".join(sorted(set(",".join(x).split(","))))),
        )
        .reset_index()
        .sort_values(["Fusion_count", "Fusion"], ascending=[False, True])
    )

    cohort_filt_fusion_level.to_csv(OUT_FILT, sep="\t", index=False)
    print(f"Wrote: {OUT_FILT} | rows: {cohort_filt_fusion_level.shape[0]}")

    # ----------------------------
    # fusion_recurrence_across_samples.tsv
    # ----------------------------
    recurrence = (
        cohort.groupby("Fusion")
        .agg(n_samples_with_fusion=("Sample", "nunique"))
        .reset_index()
        .sort_values(["n_samples_with_fusion", "Fusion"], ascending=[False, True])
    )
    recurrence.to_csv(OUT_RECURRENCE, sep="\t", index=False)
    print(f"Wrote: {OUT_RECURRENCE} | rows: {recurrence.shape[0]}")

    # ----------------------------
    # quick prints
    # ----------------------------
    print("\n=== Quick sanity ===")
    print("Total raw rows (sum of caller rows):", int(len(arriba) + len(fc) + len(sf)))
    print("Unique (Sample+Fusion) rows:", int(cohort.shape[0]))
    print("Unique fusions (Fusion):", int(recurrence.shape[0]))
    print("Concordant >=2 callers (Fusion-level rows):", int(cohort_filt_fusion_level.shape[0]))
    print("Concordant 3 callers (Sample-level rows):", int((cohort["N_callers"] == 3).sum()))
    if not recurrence.empty:
        print("Top recurrent fusion:", recurrence.iloc[0]["Fusion"])
        print("Top recurrent #samples:", int(recurrence.iloc[0]["n_samples_with_fusion"]))


if __name__ == "__main__":
    main()


[LOAD] arriba        rows=725
[LOAD] fusioncatcher rows=721
[LOAD] starfusion    rows=235

Wrote: /ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports/caller_sample_sanity.tsv | rows: 31
Wrote: /ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports/cohort.fusionreport.tsv | rows: 1354
Wrote: /ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports/cohort.filtered.fusionreport.tsv | rows: 129
Wrote: /ix1/alee/LO_LAB/Personal/Rahul/pILC/RNASeq/RNAfusion/FusionReports/fusion_recurrence_across_samples.tsv | rows: 1019

=== Quick sanity ===
Total raw rows (sum of caller rows): 1681
Unique (Sample+Fusion) rows: 1354
Unique fusions (Fusion): 1019
Concordant >=2 callers (Fusion-level rows): 129
Concordant 3 callers (Sample-level rows): 71
Top recurrent fusion: ELF1--ELF1(41522),WBP4(671)
Top recurrent #samples: 21
